# 2. 이미지 정보 추출 — 상품 이미지 분석 / 객체 인식 / 속성 추출 / JSON 출력

이커머스에서 가장 활용도가 높은 시나리오인 **상품 이미지 분석**을 중심으로,
"카테고리 분류 → 객체 나열 → 속성 추출 → 하나의 JSON"까지 단계적으로 확장합니다.

In [ ]:
from vlm_utils import ask, ask_json, show

PRODUCT = "data/product.png"
show(PRODUCT)

## 상품 이미지 분석
프롬프트에 **쇼핑몰 카테고리 체계**를 함께 제시하면 분류 정확도가 올라갑니다.

In [ ]:
category = ask(
    PRODUCT,
    "이 상품 이미지를 분석해서 아래 카테고리 중 하나로만 답하고, "
    "이유를 한 문장으로 덧붙여줘.\n"
    "카테고리: [의류, 신발, 가방, 액세서리, 가전, 기타]",
)
print(category)

## 객체 인식 (Object Recognition)

In [ ]:
objects = ask(
    PRODUCT,
    "이미지에 보이는 모든 객체를 나열하고, 각 객체의 대략적인 위치"
    "(왼쪽 위/가운데/오른쪽 아래 등)를 함께 알려줘.",
)
print(objects)

## 속성 추출 (Attribute Extraction)
추출할 속성 목록을 미리 정의해주고, 이미지에 없는 속성은
**"unknown"으로 표기**하도록 지시해 근거 없는 추측을 막습니다.
(이 상품 이미지에는 "넥라인"이 명확히 보이지 않으므로 unknown이 나오는지 확인해보세요.)

In [ ]:
attributes = ask_json(
    PRODUCT,
    '''
다음 속성만 추출해서 JSON으로만 답해: color, material, pattern, neckline, sleeve_length.
해당 속성이 이미지에서 명확히 확인되지 않으면 값으로 "unknown"을 넣어.
''',
)
print(attributes)

## JSON 출력으로 파이프라인 통합
카테고리 + 객체 + 속성을 **하나의 스키마**로 합쳐 한 번에 뽑아봅니다.
이렇게 나온 JSON은 DB 적재·검색 인덱싱·추천 시스템 입력으로 바로 활용할 수 있습니다.

In [ ]:
combined_schema = '''
아래 JSON 스키마로만 답해. 다른 텍스트는 출력하지 마.
{
  "category": "string",
  "objects": ["string"],
  "attributes": {
    "color": "string",
    "material": "string",
    "pattern": "string"
  }
}
'''
combined = ask_json(PRODUCT, combined_schema)
print(combined)
assert "category" in combined and "attributes" in combined